<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/04-Plotting-for-Astronomy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/mjcowley/Python_Intro/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics Research Group" width="400"/>

# Part 4: Plotting for Astronomy

**QUT Astrophysics Research Group (QARG) Python Training**

Group leads: Dr Michael Cowley and Dr Rossana Ruggeri
Author: Michael Cowley
Last updated: 18 Oct 2026

## Overview

A good figure is often the most important part of a paper, and making one is a skill you'll use every week of your project. The first thing your supervisor will ask to see is a plot.

This part covers **Matplotlib**, the standard plotting library in Python, using the ZFOURGE data from Part 3. By the end you'll have made several of the figures that appear in almost every galaxy evolution paper.

This notebook takes about **3 hours**.

### By the end of Part 4 you'll be able to

- Build figures with Matplotlib's figure and axes interface, and save them for a paper
- Make line plots, histograms and scatter plots, with log axes where they're needed
- Show large datasets clearly with transparency, density plots and colour bars
- Plot error bars, including asymmetric ones
- Lay out multi-panel figures
- Display astronomical images with sensible stretches and sky coordinates, and overlay catalogues on them
- Recognise what makes a figure clear, honest and accessible

## Setup

As in Part 3, `get_data()` downloads the course data files the first time you use them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy import constants as const
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ZScaleInterval, simple_norm
from astropy.wcs import WCS

DATA_URL = "https://raw.githubusercontent.com/mjcowley/Python_Intro/main/data/"


def get_data(filename):
    """Download a course data file (only the first time) and return its local path."""
    return download_file(DATA_URL + filename, cache=True)

Next, load and combine the ZFOURGE catalogues exactly as you did in Part 3, and add the columns we'll plot. If any of this is unfamiliar, have another look at Part 3, Sections 2 to 4.

In [ ]:
df = pd.read_csv(get_data("zfourge_cdfs.csv"), na_values=[-99])
eazy = Table.read(get_data("zfourge_cdfs_eazy.fits"), hdu="EAZY").to_pandas().replace(-99, np.nan)
fast = pd.read_csv(get_data("zfourge_cdfs.fout"), sep=r"\s+", comment="#", na_values=[-99],
                   names=["id", "z", "ltau", "metal", "lage", "Av", "lmass", "lsfr", "lssfr", "la2t", "chi2"])

cat = pd.merge(df, eazy[["id", "z_peak", "l68", "u68", "f_U", "f_V", "f_J"]], on="id")
cat = pd.merge(cat, fast[["id", "z", "lmass", "lsfr", "Av"]].rename(columns={"z": "z_best"}), on="id")

# Keep the objects recommended for science, and add magnitudes and rest-frame colours
gal = cat[cat["use"] == 1].copy()
gal["Ks_mag"] = 25 - 2.5 * np.log10(gal["f_Ks"].where(gal["f_Ks"] > 0))
gal["u_v"] = -2.5 * np.log10(gal["f_U"] / gal["f_V"])
gal["v_j"] = -2.5 * np.log10(gal["f_V"] / gal["f_J"])
gal["quiescent"] = (gal["u_v"] > 1.3) & (gal["v_j"] < 1.6) & (gal["u_v"] > 0.88 * gal["v_j"] + 0.59)

print(f"{len(gal)} galaxies, {gal['quiescent'].sum()} of them quiescent")

---
## 1. Anatomy of a Figure

In Parts 2 and 3 you used quick commands like `plt.plot()` and `plt.xlabel()`. These are fine for a quick look, but for real figures you should use Matplotlib's **object-oriented** interface. You create a **figure** (the whole canvas) and one or more **axes** (each individual plot), then call methods on the axes:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

z = np.linspace(0, 5, 100)
ax.plot(z, 1 / (1 + z), label="Scale factor $a = 1/(1+z)$")

ax.set_xlabel("Redshift $z$")
ax.set_ylabel("Scale factor")
ax.set_title("How big was the Universe?")
ax.legend()
plt.show()

Why bother? Once a figure has more than one panel, you need to say *which* axes you're adding to, and every method lives in one place: `ax.set_xlabel()`, `ax.set_xlim()`, `ax.set_xscale()`, `ax.legend()` and so on. You'll also find this style in almost every example online.

Notice the `$...$` in the labels. Matplotlib understands LaTeX maths, which you'll need for symbols like $M_\odot$, $\lambda$ and subscripts.

### Saving figures

Use `fig.savefig()`. For papers, save as **PDF**, which is a vector format that stays sharp at any size. For slides and websites use PNG with a high `dpi`. `bbox_inches="tight"` trims excess white space.

In [ ]:
fig.savefig("scale_factor.pdf", bbox_inches="tight")
fig.savefig("scale_factor.png", dpi=200, bbox_inches="tight")

---
## 2. Line Plots and Log Axes

Many astronomical quantities span many orders of magnitude, so log axes are everywhere. Let's plot blackbody curves (from Part 2) for stars of different temperatures.

The colours come from a **colour map**. Since temperature is an ordered quantity, we use a *sequential* colour map, which goes smoothly from light to dark, rather than a rainbow of unrelated colours.

In [ ]:
def planck(wavelength, T):
    """Blackbody spectral radiance per unit wavelength (per steradian omitted)."""
    x = const.h * const.c / (wavelength * const.k_B * T)
    return (2 * const.h * const.c**2 / wavelength**5 / (np.exp(x) - 1)).to(u.W / u.m**2 / u.nm)


wavelength = np.logspace(1, 5, 500) * u.nm         # 10 nm to 100 microns
temperatures = [3000, 5772, 10000, 40000]

cmap = plt.get_cmap("plasma")

fig, ax = plt.subplots(figsize=(7, 4.5))
for i, T in enumerate(temperatures):
    colour = cmap(i / len(temperatures))           # pick evenly spaced colours along the colour map
    ax.plot(wavelength, planck(wavelength, T * u.K), color=colour, label=f"{T:,} K")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1e2, 1e12)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel(r"$B_\lambda$ (W m$^{-2}$ nm$^{-1}$)")
ax.legend(title="Temperature")
plt.show()

On linear axes, the cool star's curve would be invisible next to the hot one. On log axes you can see all of them, and Wien's law shows up as the peak moving steadily to shorter wavelengths.

Notice the `r` before the label string. It makes a **raw string**, so the backslashes in LaTeX commands like `\lambda` aren't treated as special characters by Python. Use raw strings for any label with LaTeX in it.

### Galaxy spectra and redshift

A galaxy's spectrum (or **spectral energy distribution**, SED) is the combined light of all its stars, gas and dust. Let's load a template SED of an S0 galaxy, a type with mostly old stars, from the [SWIRE library](https://ui.adsabs.harvard.edu/abs/2007ApJ...663...81P). It's a two-column text file of wavelength in Å and flux $F_\lambda$.

In [ ]:
sed = pd.read_csv(get_data("swire_s0_template.sed"), sep=r"\s+", names=["wavelength", "flux"])
sed["wavelength_um"] = sed["wavelength"] / 1e4        # Å to microns

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sed["wavelength_um"], sed["flux"], color="k", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.1, 20)
ax.set_ylim(1e-3, 3)
ax.set_xlabel(r"Rest-frame wavelength ($\mu$m)")
ax.set_ylabel(r"$F_\lambda$ (normalised)")
plt.show()

The sharp drop near 0.4 µm is the **4000 Å break**, caused by absorption in the atmospheres of old stars. It's one of the main features photometric redshift codes like EAZY lock on to.

When a galaxy is at redshift $z$, every wavelength is stretched by a factor of $(1 + z)$, so features move to longer wavelengths.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Redshifting a Galaxy
  </h1>
</div>

1. On one set of log-log axes, plot the S0 template as it would be observed at $z = 0$, $0.5$, $1.5$ and $3$. Multiply the wavelengths by $(1 + z)$. Use a sequential colour map for the four redshifts, and add a legend.
2. Mark the wavelengths of some filters with vertical dashed lines using `ax.axvline()`: r (0.62 µm), J (1.25 µm), $K_s$ (2.15 µm) and IRAC 3.6 µm. Label each one using `ax.text()`.
3. Set the x limits to 0.3 to 10 µm.
4. Which filter does the 4000 Å break fall in at $z = 1.5$? At $z = 3$, why can't optical surveys alone tell you much about an old galaxy's stars?

*Hint: `ax.text(x, y, "label", rotation=90)` puts text at a data position. The [Redshift Explorer](https://mjcowley.github.io/astro_demos/redshift-explorer/) demo lets you do this interactively.*

In [ ]:
# your code here

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
redshifts = [0, 0.5, 1.5, 3]
filters = {"r": 0.62, "J": 1.25, "Ks": 2.15, "IRAC 3.6": 3.6}
cmap = plt.get_cmap("viridis")

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, z in enumerate(redshifts):
    ax.plot(sed["wavelength_um"] * (1 + z), sed["flux"], color=cmap(i / len(redshifts)), lw=1.2, label=f"z = {z}")

for name, wl in filters.items():
    ax.axvline(wl, color="0.5", ls="--", lw=1)
    ax.text(wl * 1.03, 1.5e-3, name, rotation=90, color="0.3", va="bottom")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.3, 10)
ax.set_ylim(1e-3, 3)
ax.set_xlabel(r"Observed wavelength ($\mu$m)")
ax.set_ylabel(r"$F_\lambda$ (normalised)")
ax.legend()
plt.show()

# At z = 1.5 the break is at 0.4 * 2.5 = 1.0 um, between r and J.
# At z = 3 it's at 1.6 um, so all the light from old stars is redwards of the optical,
# which is why ZFOURGE needed deep near-infrared imaging.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 3. Histograms

Histograms show how a quantity is distributed. The main choice you make is the **bins**: too few and you hide structure, too many and you only see noise. Always try a few.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, n_bins in zip(axes, [5, 40, 400]):
    ax.hist(gal["z_best"], bins=n_bins)
    ax.set_title(f"{n_bins} bins")
    ax.set_xlabel("Redshift")

axes[0].set_ylabel("Number of galaxies")
plt.tight_layout()       # stop the panels' labels overlapping
plt.show()

It's better to give the bin **edges** yourself, so you know exactly what each bin covers and can use the same bins for different samples. Comparing two samples is easier with `histtype="step"`, which draws outlines instead of filled bars, and `density=True`, which scales each histogram to unit area so samples of different sizes can be compared.

In [ ]:
bins = np.arange(0, 6.01, 0.2)
has_spec = gal["z_spec"].notna()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(gal["z_best"], bins=bins, histtype="step", lw=2, density=True, label=f"All ({len(gal)})")
ax.hist(gal.loc[has_spec, "z_spec"], bins=bins, histtype="step", lw=2, density=True, label=f"With spec-z ({has_spec.sum()})")
ax.set_xlabel("Redshift")
ax.set_ylabel("Probability density")
ax.legend()
plt.show()

The galaxies with spectroscopic redshifts are concentrated at lower redshift. Spectroscopy needs bright targets, so spectroscopic samples are biased towards nearby and luminous galaxies. This is why surveys like ZFOURGE rely on photometric redshifts.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Number Counts
  </h1>
</div>

The number of galaxies per magnitude bin, called the **number counts**, tells you how deep a survey goes.

1. Plot a histogram of `Ks_mag` for `gal`, with bins 0.25 magnitudes wide from 16 to 28. Use a log scale on the y axis.
2. The counts should rise steadily towards fainter magnitudes, then turn over where the survey starts missing faint galaxies. Use `np.histogram()` to find the magnitude of the bin with the most galaxies, and store the centre of that bin in `peak_mag`.
3. Mark `peak_mag` on your plot with a vertical line, and label your axes.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert 24 < peak_mag < 25.5, "peak_mag should be where the counts turn over, around Ks = 24.5 to 25."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
bins = np.arange(16, 28.01, 0.25)
mags = gal["Ks_mag"].dropna()

counts, edges = np.histogram(mags, bins=bins)
centres = (edges[:-1] + edges[1:]) / 2
peak_mag = centres[np.argmax(counts)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(mags, bins=bins, histtype="stepfilled", alpha=0.7)
ax.axvline(peak_mag, color="k", ls="--", label=f"Peak at Ks = {peak_mag:.2f}")
ax.set_yscale("log")
ax.set_xlabel("$K_s$ (AB mag)")
ax.set_ylabel("Number of galaxies per 0.25 mag")
ax.legend()
plt.show()
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 4. Scatter Plots and Crowded Data

A scatter plot shows how two quantities relate. With thousands of points, though, a default scatter plot turns into a solid blob:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(gal["z_best"], gal["lmass"])
ax.set_xlabel("Redshift")
ax.set_ylabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_title("Too crowded")
plt.show()

There are three common fixes:

1. **Small, transparent markers**: `s` sets the marker size and `alpha` the transparency, so dense regions build up darker.
2. **A 2D histogram** or **hexbin**: count the points in each cell and show the counts as colour. A log colour scale stops the densest cell washing out everything else.
3. **Contours** of the density, often drawn over the faintest points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

axes[0].scatter(gal["z_best"], gal["lmass"], s=2, alpha=0.2, color="k", rasterized=True)
axes[0].set_title("Small, transparent markers")

hb = axes[1].hexbin(gal["z_best"], gal["lmass"], gridsize=50, bins="log", mincnt=1, cmap="viridis")
fig.colorbar(hb, ax=axes[1], label="Number of galaxies")
axes[1].set_title("Hexbin with log counts")

for ax in axes:
    ax.set_xlabel("Redshift")
    ax.set_xlim(0, 5)
    ax.set_ylim(6, 12)
axes[0].set_ylabel(r"$\log_{10}(M_*/M_\odot)$")
plt.tight_layout()
plt.show()

Now the structure is clear. The lower edge of the cloud rises with redshift because we can only detect lower-mass galaxies nearby, which is the Malmquist bias from Part 3.

`rasterized=True` is a useful trick for papers. A PDF with 13,000 individual vector points is slow to open, so this stores just the points as an image while keeping the axes and text sharp.

### Colour as a third dimension

You can also colour points by a third quantity with `c=` and a colour map, and add a **colour bar** to show the scale.

In [ ]:
sub = gal[(gal["z_best"] > 1) & (gal["z_best"] < 1.5) & (gal["lmass"] > 8.5)]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sc = ax.scatter(sub["lmass"], sub["lsfr"], c=sub["Av"], s=6, cmap="magma_r", vmin=0, vmax=3)
fig.colorbar(sc, ax=ax, label="Dust attenuation $A_V$ (mag)")
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel(r"$\log_{10}(\mathrm{SFR} / M_\odot\,\mathrm{yr}^{-1})$")
ax.set_ylim(-4, 3)
ax.set_title("$1 < z < 1.5$")
plt.show()

Most galaxies follow a trend of higher star formation rate at higher stellar mass, called the **star-forming main sequence**, and more massive galaxies are also dustier. The points far below it are galaxies that have largely stopped forming stars.

These SFRs come from fitting the galaxies' SEDs with FAST, which gives quite noisy SFRs, so the sequence looks broader here than in papers. Studies of the main sequence usually measure SFRs from ultraviolet plus infrared light instead, as [Tomczak et al. (2016)](https://ui.adsabs.harvard.edu/abs/2016ApJ...817..118T) did with ZFOURGE.

You might also notice the faint diagonal **stripes** of points. They aren't real. FAST fits each galaxy with models from a grid of ages and star formation histories, and galaxies that get the same best-fit model have exactly the same SFR per unit mass, so they line up. Learning to recognise artefacts like this, which come from the analysis rather than the Universe, is an important skill.

Setting `vmin` and `vmax` fixes the colour scale, so the same colour means the same value in every figure you make.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: The Main Sequence and Quiescent Galaxies
  </h1>
</div>

Using `sub` from the cell above:

1. Plot SFR against stellar mass, showing star-forming and quiescent galaxies (from the `quiescent` column) in different colours, with a legend. Use `"tab:blue"` for star-forming and `"tab:red"` for quiescent. Plot the star-forming galaxies first, so the quiescent ones are drawn on top.
2. Do the UVJ-quiescent galaxies sit below the main sequence, as you'd expect?
3. Calculate the median SFR of star-forming galaxies in mass bins 0.25 dex wide from 9 to 11.5, and draw it as a line over the points. Store the medians in an array called `median_sfr`. *Hint: use `pd.cut()` and `groupby()` from Part 3, or a loop.*

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert len(median_sfr) == 10, "There should be 10 mass bins between 9 and 11.5."
assert median_sfr[4] > median_sfr[0] + 0.5, "The median SFR of star-forming galaxies should rise from 10^9 to 10^10 solar masses. Check you used only star-forming galaxies."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
sf = sub[~sub["quiescent"]]
q = sub[sub["quiescent"]]

edges = np.arange(9, 11.51, 0.25)
centres = (edges[:-1] + edges[1:]) / 2
median_sfr = sf.groupby(pd.cut(sf["lmass"], edges), observed=False)["lsfr"].median().to_numpy()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(sf["lmass"], sf["lsfr"], s=5, alpha=0.4, color="tab:blue", label=f"Star-forming ({len(sf)})")
ax.scatter(q["lmass"], q["lsfr"], s=8, alpha=0.8, color="tab:red", label=f"Quiescent ({len(q)})")
ax.plot(centres, median_sfr, color="k", lw=2, label="Median, star-forming")
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel(r"$\log_{10}(\mathrm{SFR} / M_\odot\,\mathrm{yr}^{-1})$")
ax.set_ylim(-4, 3)
ax.set_title("$1 < z < 1.5$")
ax.legend(loc="lower right")
plt.show()
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 5. Astronomical Conventions and Error Bars

### Inverted axes for magnitudes

Because brighter objects have *smaller* magnitudes, astronomers flip magnitude axes so that bright is at the top. The classic example is the Hertzsprung-Russell diagram. Let's make one from the Gaia stars in our field, using only stars with precise parallaxes. The absolute magnitude comes from the parallax $\varpi$ in milliarcseconds:

$$M_G = G + 5\log_{10}(\varpi / 100)$$

In [ ]:
gaia = pd.read_csv(get_data("gaia_dr3_cdfs.csv"))
good_plx = gaia[gaia["parallax"] / gaia["parallax_error"] > 5].copy()
good_plx["abs_g"] = good_plx["phot_g_mean_mag"] + 5 * np.log10(good_plx["parallax"] / 100)

fig, ax = plt.subplots(figsize=(4.5, 5))
ax.scatter(good_plx["bp_rp"], good_plx["abs_g"], s=15, color="k")
ax.invert_yaxis()                       # bright at the top
ax.set_xlabel(r"$G_\mathrm{BP} - G_\mathrm{RP}$ (mag)")
ax.set_ylabel(r"$M_G$ (mag)")
ax.set_title(f"Gaia stars towards CDFS ({len(good_plx)})")
plt.show()

With about a hundred stars it's a sparse HR diagram, but you can see the main sequence running from hotter, brighter stars at top left to cool, faint red dwarfs at bottom right. The two isolated points at bottom left are **white dwarfs**: hot, but tiny and faint. CDFS was chosen for deep surveys because it's a dark, empty patch of sky away from the Milky Way's disc, so there are very few stars in the way.

### Error bars

Every measurement has an uncertainty, and a plot without error bars can't be properly interpreted. `ax.errorbar()` draws points with error bars. For **asymmetric** errors, like the 68% range of a photometric redshift, pass a 2-row array: the lower errors first, then the upper ones.

In [ ]:
spec = gal[gal["z_spec"].notna() & (gal["z_spec"] < 4)]

# Distances from z_peak down to l68 and up to u68. Clip at zero, since z_peak occasionally falls
# just outside its own 68% range, and Matplotlib refuses negative error bars.
lower = np.clip(spec["z_peak"] - spec["l68"], 0, None)
upper = np.clip(spec["u68"] - spec["z_peak"], 0, None)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.errorbar(spec["z_spec"], spec["z_peak"], yerr=[lower, upper],
            fmt="o", ms=3, color="tab:blue", ecolor="0.7", elinewidth=0.8, alpha=0.8)
ax.plot([0, 4], [0, 4], color="k", lw=1, label="$z_\\mathrm{phot} = z_\\mathrm{spec}$")
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect("equal")                  # one unit on x is the same length as one unit on y
ax.set_xlabel("Spectroscopic redshift")
ax.set_ylabel("Photometric redshift")
ax.legend(loc="upper left")
plt.show()

Most galaxies lie close to the one-to-one line, but there are clear **outliers** far from it, like galaxy 13914 from Part 3. You'll put numbers on how good the photometric redshifts are in Part 5.

`ax.set_aspect("equal")` matters for any plot comparing two measurements of the same thing. Without it, a stretched axis can make agreement look better or worse than it really is.

### Shaded bands

For a trend with an uncertainty or a spread, a shaded band from `ax.fill_between()` is cleaner than lots of error bars. Here's the median rest-frame $U-V$ colour as a function of stellar mass, with a band covering the 16th to 84th percentiles:

In [ ]:
edges = np.arange(8.5, 11.51, 0.25)
centres = (edges[:-1] + edges[1:]) / 2
groups = sub.groupby(pd.cut(sub["lmass"], edges), observed=False)["u_v"]

median = groups.median()
p16 = groups.quantile(0.16)
p84 = groups.quantile(0.84)

fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(centres, p16, p84, color="tab:blue", alpha=0.3, lw=0, label="16th to 84th percentile")
ax.plot(centres, median, color="tab:blue", lw=2, label="Median")
ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel("Rest-frame $U-V$ (mag)")
ax.set_title("$1 < z < 1.5$")
ax.legend(loc="upper left")
plt.show()

More massive galaxies are redder on average, partly because they're more likely to be quiescent and partly because they're dustier. Always say in the caption or legend what a band or error bar represents. A standard deviation, a standard error and a percentile range all mean different things.

---
## 6. Multi-Panel Figures

`plt.subplots(nrows, ncols)` makes a grid of axes, returned as a 2D array. `sharex` and `sharey` link the axis ranges of the panels, so they're directly comparable, and Matplotlib hides the repeated tick labels.

Here's a 2 × 2 grid with a different redshift range in each panel. `axes.flat` lets you loop over all the panels in a grid as if it were a flat list.

In [ ]:
z_bins = [(0.5, 1.0), (1.0, 1.5), (1.5, 2.0), (2.0, 3.0)]

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)

for ax, (z_lo, z_hi) in zip(axes.flat, z_bins):
    in_bin = gal[(gal["z_best"] > z_lo) & (gal["z_best"] < z_hi)]
    ax.hist(in_bin["lmass"], bins=np.arange(7, 12.01, 0.25), color="0.6")
    ax.text(0.05, 0.9, f"${z_lo} < z < {z_hi}$", transform=ax.transAxes)

for ax in axes[1, :]:
    ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
for ax in axes[:, 0]:
    ax.set_ylabel("Number")

plt.tight_layout()
plt.show()

`transform=ax.transAxes` places the text in **axes coordinates**, where (0, 0) is the bottom left of the panel and (1, 1) the top right, instead of data coordinates. That keeps labels in the same spot in every panel, whatever the data ranges are.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: UVJ Diagrams Across Cosmic Time
  </h1>
</div>

Make the classic figure: a UVJ diagram in four redshift bins.

1. Make a 2 × 2 grid with shared axes, using the redshift bins in `z_bins` above. Only include galaxies with $\log_{10}(M_*/M_\odot) > 9.5$.
2. In each panel, plot $U-V$ against $V-J$, with star-forming galaxies in blue and quiescent ones in red. Use small, semi-transparent markers.
3. Draw the quiescent boundary in each panel. It's horizontal at $U-V = 1.3$ up to $V-J = 0.81$, then follows $U-V = 0.88(V-J) + 0.59$ up to $V-J = 1.6$, then goes straight up. *Hint: you can draw it with a single `ax.plot()` using the corner points `(-1, 1.3)`, `(0.81, 1.3)`, `(1.6, 1.998)` and `(1.6, 3)`.*
4. Label each panel with its redshift range and the quiescent fraction in that bin.
5. Set the limits to $-0.5 < V-J < 2.5$ and $0 < U-V < 2.5$, label the outer axes, and save the figure as `uvj_redshift.pdf`.

How does the quiescent fraction of massive galaxies change from $z \sim 2.5$ to $z \sim 0.75$?

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
import os
assert os.path.exists("uvj_redshift.pdf"), "Save your figure as uvj_redshift.pdf."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
boundary_vj = [-1, 0.81, 1.6, 1.6]
boundary_uv = [1.3, 1.3, 0.88 * 1.6 + 0.59, 3]

fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)

for ax, (z_lo, z_hi) in zip(axes.flat, z_bins):
    in_bin = gal[(gal["z_best"] > z_lo) & (gal["z_best"] < z_hi) & (gal["lmass"] > 9.5)]
    sf = in_bin[~in_bin["quiescent"]]
    q = in_bin[in_bin["quiescent"]]

    ax.scatter(sf["v_j"], sf["u_v"], s=3, alpha=0.4, color="tab:blue", rasterized=True)
    ax.scatter(q["v_j"], q["u_v"], s=3, alpha=0.6, color="tab:red", rasterized=True)
    ax.plot(boundary_vj, boundary_uv, color="k", lw=1.5)

    ax.text(0.05, 0.92, f"${z_lo} < z < {z_hi}$", transform=ax.transAxes)
    ax.text(0.05, 0.84, f"Quiescent: {in_bin['quiescent'].mean():.0%}", transform=ax.transAxes)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(0, 2.5)
for ax in axes[1, :]:
    ax.set_xlabel("$V-J$ (mag)")
for ax in axes[:, 0]:
    ax.set_ylabel("$U-V$ (mag)")

plt.tight_layout()
fig.savefig("uvj_redshift.pdf", bbox_inches="tight")
plt.show()
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

The quiescent fraction rises steadily towards lower redshift, from about 6% at $2 < z < 3$ to about 35% at $0.5 < z < 1$, and the quiescent region fills up. Over these few billion years, more and more galaxies switched off their star formation. Understanding why is a big part of what our group works on.

---
## 7. Displaying Images

Astronomical images have a huge **dynamic range**. The centre of a bright galaxy can be thousands of times brighter than the faint galaxies around it. On a linear scale you see only the brightest objects, so we use a **stretch** that compresses the bright end. Let's compare a few on the Legacy Survey image of CDFS.

In [ ]:
image_path = get_data("legacy_cdfs_r.fits")
with fits.open(image_path) as hdul:
    image = hdul[0].data
    header = hdul[0].header
wcs = WCS(header)

norms = {
    "Linear, min to max": simple_norm(image, "linear"),
    "Linear, 99.5% clip": simple_norm(image, "linear", percent=99.5),
    "Asinh": simple_norm(image, "asinh", percent=99.5),
    "ZScale": ImageNormalize(image, interval=ZScaleInterval()),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, (title, norm) in zip(axes, norms.items()):
    ax.imshow(image, origin="lower", cmap="gray_r", norm=norm)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

- **Linear from min to max** shows only the very brightest pixels.
- **Clipping** at the 99.5th percentile helps, but bright galaxies still saturate.
- **Asinh** is linear for faint pixels and logarithmic for bright ones, so it shows faint and bright objects at once. It's a good default for display.
- **ZScale** picks limits around the sky background, which is ideal for spotting faint objects and checking image quality. It's what DS9 uses by default.

Using `gray_r` (reversed grey) makes the sky white and objects dark, which reproduces well in print.

### Sky coordinates on the axes

Passing a WCS as the `projection` when you create the axes labels them with RA and Dec instead of pixel numbers. You can then plot catalogue positions directly in sky coordinates with `transform=ax.get_transform("world")`.

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=wcs)
ax.imshow(image, origin="lower", cmap="gray_r", norm=simple_norm(image, "asinh", percent=99.5))

ax.coords[0].set_axislabel("Right Ascension (J2000)")
ax.coords[1].set_axislabel("Declination (J2000)")
ax.coords.grid(color="0.6", ls=":", alpha=0.6)

ax.scatter(gaia["ra"], gaia["dec"], transform=ax.get_transform("world"),
           s=200, facecolor="none", edgecolor="tab:orange", lw=1.5, label="Gaia DR3 sources")

ax.set_xlim(-0.5, image.shape[1] - 0.5)     # keep the view on the image (WCS axes limits are in pixels)
ax.set_ylim(-0.5, image.shape[0] - 0.5)
ax.legend(loc="upper right")
plt.show()

`facecolor="none"` draws hollow circles, so you can still see the object underneath.

The Gaia catalogue covers a much larger area than the image. Without the `set_xlim()` and `set_ylim()` lines, Matplotlib would zoom out to show every point, and the image would shrink to a small square in the middle. Try commenting them out to see.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Galaxies on the Sky
  </h1>
</div>

1. Make the same image with RA and Dec axes, but this time overlay the ZFOURGE galaxies from `gal` with $\log_{10}(M_*/M_\odot) > 10$ and $z_\mathrm{best} < 2$. Use hollow circles, blue for star-forming and red for quiescent, with a legend.
2. Remember to set the axis limits back to the image, as in the example above.
3. Do the quiescent galaxies look different from the star-forming ones in this r-band image? Think back to the end of Part 3, Section 5.

In [ ]:
# your code here

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
massive = gal[(gal["lmass"] > 10) & (gal["z_best"] < 2)]

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=wcs)
ax.imshow(image, origin="lower", cmap="gray_r", norm=simple_norm(image, "asinh", percent=99.5))

for is_q, colour, label in [(False, "tab:blue", "Star-forming"), (True, "tab:red", "Quiescent")]:
    pts = massive[massive["quiescent"] == is_q]
    ax.scatter(pts["ra"], pts["dec"], transform=ax.get_transform("world"),
               s=150, facecolor="none", edgecolor=colour, lw=1.5, label=label)

ax.set_xlim(-0.5, 699.5)
ax.set_ylim(-0.5, 699.5)
ax.coords[0].set_axislabel("Right Ascension (J2000)")
ax.coords[1].set_axislabel("Declination (J2000)")
ax.set_title(r"ZFOURGE galaxies with $\log M_* > 10$, $z < 2$")
ax.legend(loc="upper right")
plt.show()

# The quiescent galaxies tend to look faint and compact in r, because at these redshifts r samples
# rest-frame blue and UV light, where old stars emit little.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

---
## 8. Making Good Figures

A figure has to be **clear**, **honest** and **readable by everyone**. Before you put a figure in a report, paper or talk, check it against this list:

| Check | Why |
|-------|-----|
| Every axis is labelled, **with units** | A number without units is meaningless |
| Text is big enough to read at the size the figure will be printed | Default Matplotlib fonts are too small for most papers and slides |
| Error bars or uncertainty bands are shown, and you say what they represent | Readers can't judge a result without its uncertainty |
| Colours are distinguishable by colour-blind readers | About 1 in 12 men have some form of colour blindness. Avoid relying on a red-green contrast alone |
| Ordered quantities use a **sequential** colour map like `viridis`, `plasma` or `magma`, never `jet` or `rainbow` | Rainbow maps have bright bands that create features that aren't in the data, and they're unreadable in greyscale |
| Each panel has **one** y axis | Two y scales on one panel invite misleading comparisons. Use two panels instead |
| Panels being compared share the same axis limits and colour scale | Otherwise the eye compares things that aren't comparable |
| Overplotted data is handled with transparency, density plots or contours | So the reader sees the distribution, not a blob |
| A legend or labels identify every colour and symbol | The reader shouldn't have to guess |

### Setting styles once

Rather than setting font sizes on every plot, set them once at the top of your notebook with `plt.rcParams`. Everything after that uses your settings.

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "xtick.direction": "in",       # ticks pointing inwards is common in astronomy journals
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

Matplotlib also has a colour-blind-friendly colour cycle you can switch to with `plt.style.use("tableau-colorblind10")`. Check the author guidelines of the journal you're submitting to for figure sizes. For a single-column figure in ApJ or MNRAS, that's about 3.3 inches wide.

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Fix This Figure
  </h1>
</div>

Run the cell below. It makes a figure that breaks most of the rules above. Then, in the empty cell, make a better version of the same figure showing the same data.

Think about: labels and units, the colour map, the second y axis, marker size and overplotting, and whether the reader can tell what's being shown.

In [ ]:
# A bad figure. Run this, then improve it in the next cell.
bad = gal[(gal["z_best"] > 0.5) & (gal["z_best"] < 2.5) & (gal["lmass"] > 9)]

fig, ax = plt.subplots()
ax.scatter(bad["lmass"], bad["lsfr"], c=bad["z_best"], cmap="jet", s=40)
ax2 = ax.twinx()
ax2.plot(np.sort(bad["lmass"]), np.linspace(0, 1, len(bad)), "g")
plt.show()

# your improved version here

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
bad = gal[(gal["z_best"] > 0.5) & (gal["z_best"] < 2.5) & (gal["lmass"] > 9)]
masses = np.sort(bad["lmass"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: one quantity per axis, a sequential colour map, small transparent markers and a labelled colour bar
sc = axes[0].scatter(bad["lmass"], bad["lsfr"], c=bad["z_best"], cmap="viridis", s=4, alpha=0.6,
                     vmin=0.5, vmax=2.5, rasterized=True)
fig.colorbar(sc, ax=axes[0], label="Redshift")
axes[0].set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
axes[0].set_ylabel(r"$\log_{10}(\mathrm{SFR} / M_\odot\,\mathrm{yr}^{-1})$")
axes[0].set_ylim(-4, 3)
axes[0].set_title("Star formation rate vs stellar mass")

# Right: the second quantity (the cumulative mass distribution) gets its own panel and its own y axis
axes[1].plot(masses, np.arange(1, len(masses) + 1) / len(masses), color="k")
axes[1].set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
axes[1].set_ylabel("Fraction of galaxies below this mass")
axes[1].set_title("Cumulative mass distribution")

fig.suptitle(f"ZFOURGE CDFS galaxies, $0.5 < z < 2.5$ ({len(bad)})")
plt.tight_layout()
plt.show()
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

<div class="practical-end" style="
  background: #f5f5f5;
  border-radius: 12px;
  padding: 20px 24px;
  text-align: center;
  border: 1px solid #ddd;
  margin: 2rem 0;
">
  <p style="font-size: 1.5rem; margin: 0 0 8px; text-align: center;">🎉</p>
  <h1 style="margin: 0 0 4px; color: #111; text-align: center;">End of Part 4</h1>
  <p style="margin: 0; color: #555; font-size: 0.9rem; text-align: center;">
    Well done! You've finished Part 4.
  </p>
</div>

<style>
@media (prefers-color-scheme: dark) {
  .practical-end { background: #1e1e1e !important; border-color: #333 !important; }
  .practical-end h1 { color: #f0f0f0 !important; }
  .practical-end p  { color: #999     !important; }
}
</style>

## What's Next?

In **Part 5** you'll put numbers on what you've been plotting: robust statistics, uncertainties, fitting models to data with $\chi^2$, and how well photometric redshifts actually work.

<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/05-Statistics-and-Fitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Want more practice?

- [Matplotlib tutorials](https://matplotlib.org/stable/tutorials/index.html), especially "Quick start guide"
- [Matplotlib gallery](https://matplotlib.org/stable/gallery/index.html), to find a plot like the one you want and adapt its code
- [Choosing colormaps in Matplotlib](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
- [Astropy: plotting images with WCSAxes](https://docs.astropy.org/en/stable/visualization/wcsaxes/)